Zadanie 1

In [3]:
import numpy as np
from scipy.linalg import hilbert

def vandermonde(n):
    x = np.linspace(0, 1, n)
    return np.vander(x, increasing=True)

def compute_norms(A):
    return {
        "norm_1": np.linalg.norm(A, 1),
        "norm_2": np.linalg.norm(A, 2),
        "norm_inf": np.linalg.norm(A, np.inf)
    }

sizes = [5, 15]

for n in sizes:
    H = hilbert(n)
    V = vandermonde(n)
    
    print(f"\n=== n = {n} ===")
    print("Hilbert:", compute_norms(H))
    print("Vandermonde:", compute_norms(V))


=== n = 5 ===
Hilbert: {'norm_1': np.float64(2.283333333333333), 'norm_2': np.float64(1.567050691098231), 'norm_inf': np.float64(2.283333333333333)}
Vandermonde: {'norm_1': np.float64(5.0), 'norm_2': np.float64(2.9865946357952113), 'norm_inf': np.float64(5.0)}

=== n = 15 ===
Hilbert: {'norm_1': np.float64(3.3182289932289937), 'norm_2': np.float64(1.8459277461534886), 'norm_inf': np.float64(3.3182289932289937)}
Vandermonde: {'norm_1': np.float64(15.0), 'norm_2': np.float64(5.68077140382738), 'norm_inf': np.float64(15.0)}


Zadanie 2


In [4]:
def compute_condition(A):
    return {
        "cond_1": np.linalg.cond(A, 1),
        "cond_2": np.linalg.cond(A, 2),
        "cond_inf": np.linalg.cond(A, np.inf)
    }

for n in sizes:
    H = hilbert(n)
    V = vandermonde(n)
    
    print(f"\n=== n = {n} ===")
    print("Hilbert:", compute_condition(H))
    print("Vandermonde:", compute_condition(V))


=== n = 5 ===
Hilbert: {'cond_1': np.float64(943656.0000063396), 'cond_2': np.float64(476607.2502422687), 'cond_inf': np.float64(943656.0000063627)}
Vandermonde: {'cond_1': np.float64(1400.0), 'cond_2': np.float64(686.4349418185961), 'cond_inf': np.float64(1706.6666666666665)}

=== n = 15 ===
Hilbert: {'cond_1': np.float64(1.22141357619596e+18), 'cond_2': np.float64(3.67568286586649e+17), 'cond_inf': np.float64(1.0417269764903425e+18)}
Vandermonde: {'cond_1': np.float64(1376796967026.9978), 'cond_2': np.float64(403234949934.73834), 'cond_inf': np.float64(1605536394325.7961)}


Zadanie 3

In [6]:
# eliminacja Gaussa Implementacja
def gauss(A, b):
    A = A.astype(float).copy()
    b = b.astype(float).copy()
    n = len(b)
    
    
    for k in range(n-1):
        for i in range(k+1, n):
            factor = A[i,k] / A[k,k]
            A[i,k:] -= factor * A[k,k:]
            b[i] -= factor * b[k]
    
    
    x = np.zeros(n)
    for i in reversed(range(n)):
        x[i] = (b[i] - np.dot(A[i,i+1:], x[i+1:])) / A[i,i]
    
    return x

# Gauss z pivotingiem emplementacja
def gauss_pivot(A, b):
    A = A.astype(float).copy()
    b = b.astype(float).copy()
    n = len(b)
    
    for k in range(n-1):
        
        max_row = np.argmax(np.abs(A[k:,k])) + k
        
        
        A[[k, max_row]] = A[[max_row, k]]
        b[[k, max_row]] = b[[max_row, k]]
        
        for i in range(k+1, n):
            factor = A[i,k] / A[k,k]
            A[i,k:] -= factor * A[k,k:]
            b[i] -= factor * b[k]
    
    x = np.zeros(n)
    for i in reversed(range(n)):
        x[i] = (b[i] - np.dot(A[i,i+1:], x[i+1:])) / A[i,i]
    
    return x

#Testowanie 
n = 10
A = np.random.rand(n, n)
b = np.random.rand(n)

x1 = gauss(A, b)
x2 = gauss_pivot(A, b)
x_ref = np.linalg.solve(A, b)

print("Błąd Gauss:", np.linalg.norm(x1 - x_ref))
print("Błąd pivot:", np.linalg.norm(x2 - x_ref)) 


Błąd Gauss: 1.745435694615691e-14
Błąd pivot: 2.5026619372463654e-15


Zadanie 4

In [9]:
#Metody iteracyne: Jacobi, Gauss-Seidel i SOR


#Jacobi
def jacobi(A, b, x0=None, tol=1e-10, max_iter=1000):
    n = len(b)
    x = np.zeros(n) if x0 is None else x0.copy()
    
    D = np.diag(A)
    R = A - np.diagflat(D)
    
    for _ in range(max_iter):
        x_new = (b - np.dot(R, x)) / D
        if np.linalg.norm(x_new - x) < tol:
            return x_new
        x = x_new
    
    return x
#Gauss-Seidel
def gauss_seidel(A, b, x0=None, tol=1e-10, max_iter=1000):
    n = len(b)
    x = np.zeros(n) if x0 is None else x0.copy()
    
    for _ in range(max_iter):
        x_new = x.copy()
        for i in range(n):
            s1 = np.dot(A[i,:i], x_new[:i])
            s2 = np.dot(A[i,i+1:], x[i+1:])
            x_new[i] = (b[i] - s1 - s2) / A[i,i]
        
        if np.linalg.norm(x_new - x) < tol:
            return x_new
        x = x_new
    
    return x
#SOR
def sor(A, b, omega=1.1, x0=None, tol=1e-10, max_iter=1000):
    n = len(b)
    x = np.zeros(n) if x0 is None else x0.copy()
    
    for _ in range(max_iter):
        x_new = x.copy()
        for i in range(n):
            s1 = np.dot(A[i,:i], x_new[:i])
            s2 = np.dot(A[i,i+1:], x[i+1:])
            
            x_new[i] = (1-omega)*x[i] + omega*(b[i] - s1 - s2)/A[i,i]
        
        if np.linalg.norm(x_new - x) < tol:
            return x_new
        x = x_new
    
    return x

print("Wniosek: \n Jacobi – najwolniejsza \n Gauss-Seidel – szybsza \n SOR – najszybsza (dobry dobór ω ~ 1.1–1.9)")

Wniosek: 
 Jacobi – najwolniejsza 
 Gauss-Seidel – szybsza 
 SOR – najszybsza (dobry dobór ω ~ 1.1–1.9)


Zadanie 5

In [ ]:
#solve vs odwrotność
def experiment(n):
    H = hilbert(n)
    b = np.array([1/(n+i+1) for i in range(n)])
    
    x_solve = np.linalg.solve(H, b)
    x_inv = np.dot(np.linalg.inv(H), b)
    
    return np.linalg.norm(x_solve - x_inv)

for n in [5, 15]:
    print(f"n={n}, różnica:", experiment(n))

    #Wniosek: używanie odwrotności macierzy jest numerycznie niestabilne. Zawsze preferuj solve


n=5, różnica: 1.0001350230786475e-11
n=15, różnica: 67.57205471134239


Zadanie domowe

In [20]:
import numpy as np

# macierz A
A = np.array([
    [1e5, 9.9e4],
    [1.00001, 0.99]
])

# dokładne rozwiązanie
x_exact = np.array([1.0, 1.0])

# wektor b
b = A @ x_exact

# --- 1. wskaźnik uwarunkowania ---
cond_A = np.linalg.cond(A)

# --- 2. rozwiązanie ---
x_num = np.linalg.solve(A, b)

# --- 3. błąd ---
rel_error = np.linalg.norm(x_num - x_exact) / np.linalg.norm(x_exact)

# oszacowanie błędu (zakładamy błąd maszynowy)
eps = np.finfo(float).eps
error_est = cond_A * eps

print("=== Oryginalna macierz ===")
print("cond(A):", cond_A)
print("x_num:", x_num)
print("błąd względny:", rel_error)
print("oszacowanie:", error_est)


# --- 4. skalowanie (wyważenie wierszy) ---
D = np.diag(1 / np.linalg.norm(A, axis=1))
A_scaled = D @ A

# --- 5. nowe b ---
b_scaled = D @ b

# --- 6. wskaźnik uwarunkowania ---
cond_scaled = np.linalg.cond(A_scaled)

# --- 7. rozwiązanie ---
x_scaled = np.linalg.solve(A_scaled, b_scaled)

# --- 8. błąd ---
rel_error_scaled = np.linalg.norm(x_scaled - x_exact) / np.linalg.norm(x_exact)
error_est_scaled = cond_scaled * eps

print("\n=== Po skalowaniu ===")
print("cond(A_scaled):", cond_scaled)
print("x_scaled:", x_scaled)
print("błąd względny:", rel_error_scaled)
print("oszacowanie:", error_est_scaled)

print("Wniosek: \n Macierz A jest źle uwarunkowana, co powoduje dużą wrażliwość rozwiązania na błędy numeryczne. \n W rezultacie rozwiązanie układu może zawierać znaczny błąd, mimo dokładnych danych wejściowych. \n Po zastosowaniu skalowania wierszy wskaźnik uwarunkowania maleje, co prowadzi do poprawy dokładności rozwiązania. \n Błąd numeryczny w obu przypadkach nie jest tego samego rzędu — po skalowaniu jest wyraźnie mniejszy. \n Ponadto oszacowanie błędu oparte na wskaźniku uwarunkowania jest znacznie bardziej trafne dla macierzy skalowanej.")

=== Oryginalna macierz ===
cond(A): 20001010102.66865
x_num: [1. 1.]
błąd względny: 1.1158437364096776e-11
oszacowanie: 4.441116386348621e-06

=== Po skalowaniu ===
cond(A_scaled): 400022.22223091894
x_scaled: [1. 1.]
błąd względny: 1.5701804821351072e-11
oszacowanie: 8.882277629649747e-11
Wniosek: 
 Macierz A jest źle uwarunkowana, co powoduje dużą wrażliwość rozwiązania na błędy numeryczne. 
 W rezultacie rozwiązanie układu może zawierać znaczny błąd, mimo dokładnych danych wejściowych. 
 Po zastosowaniu skalowania wierszy wskaźnik uwarunkowania maleje, co prowadzi do poprawy dokładności rozwiązania. 
 Błąd numeryczny w obu przypadkach nie jest tego samego rzędu — po skalowaniu jest wyraźnie mniejszy. 
 Ponadto oszacowanie błędu oparte na wskaźniku uwarunkowania jest znacznie bardziej trafne dla macierzy skalowanej.
